# Google QEC data discovery

The source ZIP stays unchanged in Bronze. These cells check registration, build the two Google Silver tables, and inspect one source file.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from quantum_lake_student.stages.register_sources import run
result = run(run_id="test-1")
print(result)

- Bronze registration verifies the three source ZIPs against the release manifest and checks their archive members.

In [ ]:
from io import BytesIO
from zipfile import ZipFile
import pandas as pd
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import read_lake_object
from quantum_lake_student.google_qec import BRONZE_OBJECT

with ZipFile(BytesIO(read_lake_object(Settings.from_environment(), BRONZE_OBJECT))) as archive:
    member_names = [
        name for name in archive.namelist()
        if name.startswith("surface_code_") and "/" in name
    ]

inventory = pd.DataFrame({
    "member": member_names,
    "experiment_id": [name.split("/", 1)[0] for name in member_names],
})
member_counts = inventory.groupby("experiment_id").size().rename("files")
display(member_counts.to_frame())
assert len(member_counts) == 5
assert member_counts.eq(15).all()

- The archive contains five experiment directories with 15 members each. Companion files include metadata, packed measurements and detector events, actual flips, and four decoder predictions.

In [ ]:
from quantum_lake_student.stages import silvergoogle
result = silvergoogle.run(run_id="test-1")
print(result)

- The Google stage produces one experiment row per directory and one shot row per aligned set of companion-file records.
- The supplied release has five experiments and 250,000 shots.

In [ ]:
from io import BytesIO
import pandas as pd
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import read_lake_object
from quantum_lake_student.google_qec import EXPERIMENT_PATH, SHOT_PATH

settings = Settings.from_environment()
exp = pd.read_parquet(BytesIO(read_lake_object(settings, EXPERIMENT_PATH)))
shots = pd.read_parquet(BytesIO(read_lake_object(settings, SHOT_PATH)))
print(len(exp))    # should be 5
print(len(shots))  # should be 250000 (5 x 50000), assuming no rejected experiments

print(exp)
print(shots.head())

- `experiment_id` links each shot to its experiment; `shot_index` locates the shot within that experiment.
- Measurement, sweep, and detector bits remain separate packed byte fields. The actual logical flip and four decoder predictions are separate binary columns.
- Each Silver row has a stable `source_record_id` for tracing back to the Bronze members.

In [ ]:
experiment_profile = exp[[
    "experiment_id", "basis", "distance", "rounds", "shots",
    "center_row", "center_col", "measurement_count", "detector_count",
]]
shot_profile = shots.groupby("experiment_id").agg(
    silver_shots=("shot_index", "size"),
    distinct_indices=("shot_index", "nunique"),
    first_index=("shot_index", "min"),
    last_index=("shot_index", "max"),
)
alignment = experiment_profile.set_index("experiment_id").join(shot_profile)
alignment["shots_align"] = (
    alignment["shots"].eq(alignment["silver_shots"])
    & alignment["shots"].eq(alignment["distinct_indices"])
    & alignment["first_index"].eq(0)
    & alignment["last_index"].eq(alignment["shots"] - 1)
)
display(alignment)
assert alignment["shots_align"].all()

- Each experiment has 50,000 distinct shot indices, from 0 through 49,999. Four experiments have distance three; one has distance five.
- The experiment-to-shot relation is one-to-many. The companion files align through the experiment directory and shot index.

In [ ]:
from io import BytesIO
from zipfile import ZipFile
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import read_lake_object
from quantum_lake_student.google_qec import BRONZE_OBJECT

zip_file = ZipFile(BytesIO(read_lake_object(
    Settings.from_environment(), BRONZE_OBJECT
)))
raw = zip_file.read("surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01")
zip_file.close()
print("Total bytes:", len(raw))
print("Repr of first 40 bytes:", repr(raw[:40]))

lines = raw.splitlines()
print("Number of lines via splitlines():", len(lines))
print("First 10 lines:", lines[:10])

- This `.01` file has 50,000 lines: one actual logical-flip value per shot. Its 100,000 bytes include the line endings.
- A prediction file must have the same number of binary lines so values stay aligned by shot index.

In [ ]:
decoder_columns = [
    "belief_matching_prediction",
    "correlated_matching_prediction",
    "pymatching_prediction",
    "tensor_network_contraction_prediction",
]
decoder_profile = pd.DataFrame({
    "decoder": decoder_columns,
    "mistakes_against_actual": [
        shots[column].ne(shots["actual_observable_flip"]).sum()
        for column in decoder_columns
    ],
})
display(decoder_profile)

- A decoder mistake is a prediction that differs from the actual logical flip. The actual flip is the label; the four predictions are separate inputs.

In [ ]:
issues = pd.read_parquet("results/part1/data_issues.parquet")
google_issues = issues.loc[issues["source_name"] == "google_qec"]
print(google_issues[["rule_id", "record_locator", "observed_value"]])

## Findings

- The supplied Google records produced no data issues in this run.
- The five experiment directories supply aligned shots, but no source identifier links these shots to a QASMBench circuit row. That cross-source row join should not be invented.